# **Lab 05.1 - Q4: Proximal Policy Optimization (PPO) on CartPole-v1**


## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

This split notebook contains the material for **Question 4 only** from Lab 05.1.
- **Part 3:** PPO (from scratch) on CartPole-v1
- **Part 4:** Stable-Baselines3 PPO baseline on CartPole-v1

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>
</div>


### Imports and Setup

The following cell configures the runtime and defines shared utilities used across all parts of this lab.

In [ ]:
# Runtime stability settings for notebook environments on macOS
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Keep CPU thread usage predictable in notebook kernels
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    from stable_baselines3 import PPO as SB3PPO, TD3
    from stable_baselines3.common.noise import NormalActionNoise
    sb3_library_available = True
except Exception:
    SB3PPO = None
    TD3 = None
    NormalActionNoise = None
    sb3_library_available = False

# Reproducibility settings
SEED = 42
FAST_MODE = True

# Set seeds for all libraries to ensure reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True, 'grid.alpha': 0.25})

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Gymnasium: {gym.__version__}')
print(f'FAST_MODE: {FAST_MODE}')

In [ ]:
def moving_average_curve(raw_values, window_size=10):
    values_array = np.asarray(raw_values, dtype=np.float32)
    if len(values_array) < window_size:
        return values_array
    kernel_array = np.ones(window_size, dtype=np.float32) / window_size
    return np.convolve(values_array, kernel_array, mode='valid')


def plot_training_curve(training_values, title_text, y_label_text='Return', window_size=10):
    plt.figure(figsize=(8.5, 3.8))
    plt.plot(training_values, alpha=0.35, label='Raw')
    averaged_values = moving_average_curve(training_values, window_size=window_size)
    if len(averaged_values) > 0:
        x_index_values = np.arange(window_size - 1, window_size - 1 + len(averaged_values))
        plt.plot(x_index_values, averaged_values, linewidth=2.0, label=f'MA({window_size})')
    plt.title(title_text)
    plt.xlabel('Episode / Epoch')
    plt.ylabel(y_label_text)
    plt.legend()
    plt.show()


def evaluate_policy_returns(environment_name, policy_action_function, episode_count=8, seed_offset_value=1000):
    episodic_return_values = []
    for episode_index in range(episode_count):
        evaluation_env_instance = gym.make(environment_name)
        current_state, _ = evaluation_env_instance.reset(seed=SEED + seed_offset_value + episode_index)
        episode_done_flag = False
        total_episode_return = 0.0
        while not episode_done_flag:
            selected_action = policy_action_function(current_state)
            next_state, reward_value, terminated_flag, truncated_flag, _ = evaluation_env_instance.step(selected_action)
            total_episode_return += reward_value
            current_state = next_state
            episode_done_flag = terminated_flag or truncated_flag
        episodic_return_values.append(total_episode_return)
        evaluation_env_instance.close()
    return np.asarray(episodic_return_values, dtype=np.float32)

## Part 3: Discrete Control via Proximal Policy Optimization (PPO)

### 3.1. PPO Policy & Value Networks (From Scratch)

**Pipeline Architecture**:
1. Initialize `cartpole_discrete_env` and inspect state/action dimensions.
2. `ppo_scratch_policy_net` emits logits over discrete actions.
3. Sample action using `torch.distributions.Categorical`.
4. Store `(state, action, log_prob, reward, done, value)` in `DiscreteRolloutBuffer`.

**Component Interactions**:
- Policy and value networks are optimized jointly in PPO updates.
- Rollout buffer supplies trajectories to GAE and clipped objective cells.

**Mathematical/Algorithmic Anchors**:
- Categorical policy: $\pi_\theta(a|s)=\text{Softmax}(\text{logits}_\theta(s))$.
- Log-probabilities from Categorical distribution support PPO ratio computation.

In [ ]:
# Probe the CartPole-v1 environment to understand its observation and action spaces, as well as reward structure
cartpole_discrete_env = 'CartPole-v1'
cartpole_probe_env = gym.make(cartpole_discrete_env)
cartpole_observation_dimension = cartpole_probe_env.observation_space.shape[0]
cartpole_action_count = cartpole_probe_env.action_space.n

print('Environment:', cartpole_discrete_env)
print('Observation dimension:', cartpole_observation_dimension)
print('Action count:', cartpole_action_count)

cartpole_probe_env.close()

In [ ]:
class DiscreteRolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.state_list = []
        self.action_list = []
        self.log_probability_list = []
        self.reward_list = []
        self.done_list = []
        self.value_list = []

    def add(self, state_value, action_value, log_probability_value, reward_value, done_value, value_estimate):
        self.state_list.append(state_value)
        self.action_list.append(int(action_value))
        self.log_probability_list.append(float(log_probability_value))
        self.reward_list.append(float(reward_value))
        self.done_list.append(float(done_value))
        self.value_list.append(float(value_estimate))


class PPOScratchNetwork(nn.Module):
    def __init__(self, state_dimension, action_count, hidden_size=128):
        super().__init__()
        self.shared_torso = nn.Sequential(
            # Architechture of the shared torso:
            # Linear (state_dimension -> hidden_size) -> Tanh -> Linear (hidden_size -> hidden_size) -> Tanh
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            
        )
        # The policy head outputs action logits for each discrete action, 
        # while the value head outputs a single scalar value estimate for the input state.
        # ppo_scratch_policy_net = Linear layer that maps the shared latent representation to action logits for each discrete action
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.ppo_scratch_policy_net = 
        # ppo_scratch_value_net = Linear layer that maps the shared latent representation to a single scalar value estimate for the input state
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        self.ppo_scratch_value_net = 

    def forward(self, state_tensor):
        # latent_tensor = output of the shared torso network for the given input state tensor
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        latent_tensor = 
        # action_logits = output of the policy head network for the latent representation, 
        # which gives the unnormalized log probabilities for each discrete action
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        action_logits = 
        # state_value = output of the value head network for the latent representation, 
        # which gives the estimated value of the input state
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        state_value = 
        return action_logits, state_value

### 3.2. Generalized Advantage Estimation (GAE) & Optimization Loop

**Pipeline Architecture**:
1. Collect on-policy rollout with current PPO parameters.
2. Compute GAE advantages and returns.
3. Re-evaluate policy logits and values on mini-batches.
4. Compute PPO clipped policy loss + value loss + entropy regularization.
5. Update parameters over multiple epochs.

**Component Interactions**:
- `DiscreteRolloutBuffer` feeds this optimization loop.
- `PPOScratchNetwork` outputs both policy logits and value estimates.

**Mathematical/Algorithmic Anchors**:
- GAE recursion for variance-reduced advantages.
- Clipped surrogate: $L^{clip}(\theta) = \mathbb{E}[\min(r_t(\theta)A_t, \text{clip}(r_t(\theta),1-\epsilon,1+\epsilon)A_t)]$.

In [ ]:
def compute_generalized_advantages(reward_sequence, done_sequence, value_sequence, gamma_value=0.99, lambda_value=0.95):
    advantage_values = []
    running_gae = 0.0
    extended_values = list(value_sequence) + [0.0]
    for transition_index in reversed(range(len(reward_sequence))):
        # td_error = reward + gamma * next_value * (1 - done) - current_value
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        td_error = 
        # GAE = td_error + gamma * lambda * (1 - done) * running_gae
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        running_gae = 
        advantage_values.insert(0, running_gae)
    advantage_array = np.asarray(advantage_values, dtype=np.float32)
    
    # return = advantage + value, which gives the target return for each state, 
    # used for training the value function in PPO.
    ### YOU NEED TO WRITE YOUR CODE BELOW ###
    return_array = 
    return advantage_array, return_array


class PPOScratchTrainer:
    def __init__(self, state_dimension, action_count, learning_rate=3e-4, clip_ratio=0.2):
        self.clip_ratio = clip_ratio
        self.policy_value_network = PPOScratchNetwork(state_dimension, action_count).to(device)
        self.network_optimizer = optim.Adam(self.policy_value_network.parameters(), lr=learning_rate)

    def sample_action(self, observation_state):
        state_tensor = torch.tensor(observation_state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            # action_logits, state_value = output of the policy-value network for the given state tensor, 
            # which includes the action logits for each discrete action and the estimated state value.
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            action_logits, state_value = 
            categorical_distribution = torch.distributions.Categorical(logits=action_logits)
            sampled_action_tensor = categorical_distribution.sample()
            sampled_log_probability = categorical_distribution.log_prob(sampled_action_tensor)
        return int(sampled_action_tensor.item()), float(sampled_log_probability.item()), float(state_value.item())

    def greedy_action(self, observation_state):
        state_tensor = torch.tensor(observation_state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action_logits, _ = self.policy_value_network(state_tensor)
        greedy_action = int(torch.argmax(action_logits, dim=1).item())
        return greedy_action

    def optimize_from_buffer(self, rollout_buffer, gamma_value=0.99, lambda_value=0.95, epochs_count=4, minibatch_size=128):
        # Convert the collected rollout data into PyTorch tensors for efficient batch processing during optimization.
        # state_tensor = tensor containing the states from the rollout buffer, converted to float32 and moved to the appropriate device (CPU or GPU).
        # action_tensor = tensor containing the actions from the rollout buffer, converted to long integers (for indexing) and moved to the appropriate device.
        # old_log_probability_tensor = tensor containing the log probabilities of the actions taken during the rollout, converted to float32 and moved to the appropriate device.
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        state_tensor = 
        action_tensor = 
        old_log_probability_tensor = 

        advantage_array, return_array = compute_generalized_advantages(
            rollout_buffer.reward_list,
            rollout_buffer.done_list,
            rollout_buffer.value_list,
            gamma_value=gamma_value,
            lambda_value=lambda_value,
        )
        
        advantage_tensor = torch.tensor(advantage_array, dtype=torch.float32, device=device)
        return_tensor = torch.tensor(return_array, dtype=torch.float32, device=device).unsqueeze(1)
        # Normalize the advantage estimates to have zero mean and unit variance, 
        # which can help stabilize training by preventing excessively large updates to the policy parameters.
        advantage_tensor = (advantage_tensor - advantage_tensor.mean()) / (advantage_tensor.std() + 1e-8)

        sample_count = state_tensor.shape[0]
        index_array = np.arange(sample_count)

        for _ in range(epochs_count):
            # Shuffle the indices of the collected samples to ensure that each optimization epoch processes the data in a different order, 
            # which can help improve generalization and prevent overfitting to specific sequences of transitions.
            ### YOU NEED TO WRITE YOUR CODE BELOW ###

            for start_index in range(0, sample_count, minibatch_size):
                
                ### batch_indices = subset of shuffled indices for the current mini-batch
                ### YOU NEED TO WRITE YOUR CODE BELOW ###
                batch_indices = 

                batch_states = state_tensor[batch_indices]
                batch_actions = action_tensor[batch_indices]
                batch_old_log_probabilities = old_log_probability_tensor[batch_indices]
                batch_advantages = advantage_tensor[batch_indices]
                batch_returns = return_tensor[batch_indices]

                batch_logits, batch_values = self.policy_value_network(batch_states)
                batch_distribution = torch.distributions.Categorical(logits=batch_logits)
                batch_new_log_probabilities = batch_distribution.log_prob(batch_actions)

                # probability_ratio = exp(new_log_probability - old_log_probability) = new_probability / old_probability
                ### YOU NEED TO WRITE YOUR CODE BELOW ###
                probability_ratio = 

                # unclipped_objective = probability_ratio * batch_advantages
                ### YOU NEED TO WRITE YOUR CODE BELOW ###
                unclipped_objective = 
                
                # clipped_objective = clamp the probability_ratio to be within [1 - clip_ratio, 1 + clip_ratio] and then multiply by batch_advantages
                ### YOU NEED TO WRITE YOUR CODE BELOW ###
                clipped_objective = 
                # - sign is used because we typically minimize the loss, 
                # but the PPO objective is formulated as a maximization problem,
                clipped_policy_loss = -torch.min(unclipped_objective, clipped_objective).mean()

                # The value loss is computed as the mean squared error between the predicted state values and the computed returns,
                value_loss = nn.MSELoss()(batch_values, batch_returns)
                # The entropy bonus is calculated as the mean entropy of the action distribution, 
                # which encourages exploration by penalizing low-entropy (i.e., more deterministic) policies.
                ### YOU NEED TO WRITE YOUR CODE BELOW ###
                entropy_bonus = 
                # The total loss for the PPO update is a combination of the clipped policy loss, the value loss (weighted by 0.5), and the entropy bonus (weighted by -0.01 to encourage exploration).
                # total_loss = clipped_policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus
                total_loss = 

                self.network_optimizer.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(self.policy_value_network.parameters(), 0.5)
                self.network_optimizer.step()

In [ ]:
ppo_scratch_trainer = PPOScratchTrainer(cartpole_observation_dimension, cartpole_action_count, learning_rate=3e-4, clip_ratio=0.2)
cartpole_rollout_buffer = DiscreteRolloutBuffer()
cartpole_training_environment = gym.make(cartpole_discrete_env)

ppo_training_epochs = 24 if FAST_MODE else 120
rollout_steps_per_epoch = 512
ppo_training_returns = []

for epoch_index in range(ppo_training_epochs):
    cartpole_rollout_buffer.clear()
    current_observation, _ = cartpole_training_environment.reset(seed=SEED + epoch_index)
    epoch_return_value = 0.0

    for _ in range(rollout_steps_per_epoch):
        selected_action, action_log_probability, value_estimate = ppo_scratch_trainer.sample_action(current_observation)
        next_observation, reward_value, terminated_flag, truncated_flag, _ = cartpole_training_environment.step(selected_action)
        done_value = float(terminated_flag or truncated_flag)
        # Store the transition data in the rollout buffer for later optimization after collecting a batch of transitions.
        cartpole_rollout_buffer.add(current_observation, selected_action, action_log_probability, reward_value, done_value, value_estimate)
        epoch_return_value += reward_value
        current_observation = next_observation

        if terminated_flag or truncated_flag:
            current_observation, _ = cartpole_training_environment.reset()

    # After collecting the specified number of rollout steps for the current epoch, 
    # we call the optimize_from_buffer method of the PPO trainer to perform multiple epochs of optimization using mini-batches of the collected data, 
    # which updates the policy and value networks based on the PPO objective.
    ppo_scratch_trainer.optimize_from_buffer(
        cartpole_rollout_buffer, gamma_value=0.99, lambda_value=0.95, epochs_count=4, minibatch_size=128
    )
    ppo_training_returns.append(epoch_return_value)

    if (epoch_index + 1) % 6 == 0:
        print(f'[PPO Scratch] Epoch {epoch_index + 1}/{ppo_training_epochs} Return={epoch_return_value:.2f}')

cartpole_training_environment.close()
plot_training_curve(ppo_training_returns, 'PPO Scratch on CartPole-v1', window_size=4)


def ppo_scratch_action_function(observation_state):
    return ppo_scratch_trainer.greedy_action(observation_state)

ppo_scratch_evaluation_returns = evaluate_policy_returns(
    cartpole_discrete_env, ppo_scratch_action_function, episode_count=12, seed_offset_value=4000
)
print('PPO Scratch mean/std:', float(ppo_scratch_evaluation_returns.mean()), float(ppo_scratch_evaluation_returns.std()))

## Part 4: Discrete Control via Stable-Baselines3 PPO

### 4.1. SB3 PPO Baseline Training

**Pipeline Architecture**:
1. Initialize SB3 PPO with MLP policy on `cartpole_discrete_env`.
2. Train baseline policy with standard PPO defaults adapted to this course lab.

**Component Interactions**:
- Baseline shares environment and evaluation harness with scratch PPO.

**Mathematical/Algorithmic Anchors**:
- PPO clipping, value updates, and entropy regularization are encapsulated in SB3 internals.

In [ ]:
if sb3_library_available:
    ppo_sb3_training_environment = gym.make(cartpole_discrete_env)
    ppo_sb3_model = SB3PPO(
        'MlpPolicy',
        ppo_sb3_training_environment,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        seed=SEED,
        verbose=0,
        device=device,
    )
    ppo_sb3_training_timesteps = 9000 if FAST_MODE else 45000
    ppo_sb3_model.learn(total_timesteps=ppo_sb3_training_timesteps, progress_bar=False)
    ppo_sb3_training_environment.close()
else:
    ppo_sb3_model = None
    print('SB3 not available: PPO baseline skipped.')

### 4.2. Comparative Evaluation

**Pipeline Architecture**:
1. Evaluate scratch PPO and SB3 PPO over shared seeded episodes.
2. Build a side-by-side result table.
3. Overlay mean-return bars with standard-deviation error bars.

**Component Interactions**:
- Evaluation helper function ensures reproducible comparison.

**Mathematical/Algorithmic Anchors**:
- Mean and variance estimates summarize policy reliability under stochastic trajectories.

In [ ]:
discrete_results_dictionary = {
    'PPO Scratch': {
        'mean_return': float(ppo_scratch_evaluation_returns.mean()),
        'std_return': float(ppo_scratch_evaluation_returns.std()),
    }
}

if ppo_sb3_model is not None:
    def ppo_sb3_action_function(observation_state):
        baseline_action, _ = ppo_sb3_model.predict(observation_state, deterministic=True)
        return int(baseline_action)

    ppo_sb3_evaluation_returns = evaluate_policy_returns(
        cartpole_discrete_env, ppo_sb3_action_function, episode_count=12, seed_offset_value=5000
    )
    discrete_results_dictionary['PPO SB3'] = {
        'mean_return': float(ppo_sb3_evaluation_returns.mean()),
        'std_return': float(ppo_sb3_evaluation_returns.std()),
    }

discrete_results_frame = pd.DataFrame(discrete_results_dictionary).T
print(discrete_results_frame)

plt.figure(figsize=(6.5, 3.5))
plt.bar(
    discrete_results_frame.index,
    discrete_results_frame['mean_return'],
    yerr=discrete_results_frame['std_return'],
    capsize=4,
)
plt.title('CartPole-v1: PPO Scratch vs PPO SB3')
plt.ylabel('Mean Return')
plt.show()

---
# CONGRATULATIONS TEAM!

This split notebook focuses only on **Question 4** for CartPole PPO:
- **PPO (Scratch)** on **CartPole-v1**
- **PPO (SB3)** as a stable reference baseline

Technical trade-off summary:
- PPO from scratch remains stable via clipped policy optimization and GAE.
- SB3 PPO provides a strong out-of-the-box reference for comparison.
- The two implementations are evaluated side by side using mean and standard deviation of returns.


### References

- Lillicrap et al., Continuous control with deep reinforcement learning (DDPG), ICLR 2016
- Fujimoto et al., Addressing Function Approximation Error in Actor-Critic Methods (TD3), ICML 2018
- Schulman et al., Proximal Policy Optimization Algorithms, arXiv:1707.06347
- Schulman et al., High-Dimensional Continuous Control Using GAE, ICLR 2016
- Gymnasium documentation: https://gymnasium.farama.org/
- Stable-Baselines3 documentation: https://stable-baselines3.readthedocs.io/
- PyTorch documentation: https://pytorch.org/

---

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn

**Last Updated**: March, 2026